In [5]:
pip install fuzzywuzzy

Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from scipy import stats

# Load your data
df = pd.read_csv('salary_data.csv')

# For demonstration, creating sample data structure
# df = pd.DataFrame({
#     'Age': [32, 28, 45, 36],
#     'Gender': ['Male', 'Female', 'Male', 'Female'],
#     'Education Level': ["Bachelor's", "Master's", 'PhD', "Bachelor's"],
#     'Job Title': ['Software Engineer', 'Data Analyst', 'Senior Manager', 'Sales Associate'],
#     'Years of Experience': [5, 3, 15, 7],
#     'Salary': [90000, 65000, 150000, 60000]
# })

In [15]:


# ============================================
# 1. DUPLICATE REMOVAL (Fuzzy Matching)
# ============================================
def fuzzy_match_duplicates(df, threshold=90):
    """Remove fuzzy duplicates based on key columns"""
    df = df.copy()

    # Create comparison string for fuzzy matching
    if all(col in df.columns for col in ['Company', 'Job Title', 'Location', 'Salary']):
        df['compare_str'] = (df['Company'].fillna('') + '|' +
                            df['Job Title'].fillna('') + '|' +
                            df['Location'].fillna('') + '|' +
                            df['Salary'].astype(str))

        to_remove = set()
        for i in range(len(df)):
            if i in to_remove:
                continue
            for j in range(i+1, len(df)):
                if j in to_remove:
                    continue
                similarity = fuzz.ratio(df.iloc[i]['compare_str'],
                                       df.iloc[j]['compare_str'])
                if similarity >= threshold:
                    to_remove.add(j)  # Keep first occurrence, remove duplicates

        df = df[~df.index.isin(to_remove)]
        df = df.drop('compare_str', axis=1)  # Only drop the comparison string

    return df

# ============================================
# 2. MISSING VALUE HANDLING
# ============================================
def handle_missing_values(df):
    """Impute missing values using median/mode strategies"""
    df = df.copy()

    # Median imputation for experience (5% missing)
    if 'Years of Experience' in df.columns:
        median_exp = df['Years of Experience'].median()
        df['Years of Experience'].fillna(median_exp, inplace=True)

    # Mode imputation for education (8% missing)
    if 'Education Level' in df.columns:
        mode_edu = df['Education Level'].mode()[0] if not df['Education Level'].mode().empty else "Bachelor's"
        df['Education Level'].fillna(mode_edu, inplace=True)

    # Remove records with missing salary (core variable)
    if 'Salary' in df.columns:
        df = df.dropna(subset=['Salary'])

    return df

# ============================================
# 3. STANDARDIZATION
# ============================================
def standardize_job_titles(title):
    """Standardize job title variations"""
    if pd.isna(title):
        return title

    title_lower = str(title).lower()

    # Software Engineer variations
    if any(x in title_lower for x in ['swe', 'se ', 'software engineer', 'software eng']):
        return 'Software Engineer'

    # Data Analyst variations
    if any(x in title_lower for x in ['data analyst', 'data analysis']):
        return 'Data Analyst'

    # Manager variations
    if 'manager' in title_lower:
        if 'senior' in title_lower or 'sr' in title_lower:
            return 'Senior Manager'
        return 'Manager'

    return title.title()

def standardize_locations(location):
    """Standardize location variations"""
    if pd.isna(location):
        return location

    loc_lower = str(location).lower()

    # San Francisco variations
    if any(x in loc_lower for x in ['sf', 'san fran', 'san francisco']):
        return 'San Francisco'

    # New York variations
    if any(x in loc_lower for x in ['ny', 'nyc', 'new york']):
        return 'New York'

    return location.title()

def standardize_companies(company):
    """Standardize company name variations"""
    if pd.isna(company):
        return company

    comp_lower = str(company).lower()

    # Meta/Facebook variations
    if any(x in comp_lower for x in ['fb', 'facebook', 'meta']):
        return 'Meta'

    # Google/Alphabet variations
    if any(x in comp_lower for x in ['google', 'alphabet']):
        return 'Google'

    return company.title()

def normalize_salary(row):
    """Convert hourly/monthly salaries to annual"""
    salary = row['Salary']

    # Check if there's a salary type column
    if 'Salary Type' in row and pd.notna(row['Salary Type']):
        salary_type = str(row['Salary Type']).lower()

        if 'hourly' in salary_type or 'hour' in salary_type:
            # Assume 40 hours/week, 52 weeks/year
            return salary * 40 * 52
        elif 'monthly' in salary_type or 'month' in salary_type:
            return salary * 12

    # If salary seems too low, might be hourly or monthly
    if salary < 1000:  # Likely hourly
        return salary * 40 * 52
    elif 1000 <= salary < 15000:  # Likely monthly
        return salary * 12

    return salary

# ============================================
# 4. FEATURE ENGINEERING
# ============================================
def extract_years_experience(exp_str):
    """Extract years of experience from various formats"""
    if pd.isna(exp_str):
        return np.nan

    # If already a number
    if isinstance(exp_str, (int, float)):
        return exp_str

    exp_lower = str(exp_str).lower()

    # Handle ranges like "3-5 years" -> average (4)
    import re
    range_match = re.search(r'(\d+)\s*-\s*(\d+)', exp_lower)
    if range_match:
        return (int(range_match.group(1)) + int(range_match.group(2))) / 2

    # Handle single numbers
    num_match = re.search(r'\d+', exp_lower)
    if num_match:
        return int(num_match.group())

    # Handle text levels
    if 'entry' in exp_lower or 'junior' in exp_lower:
        return 1
    elif 'mid' in exp_lower or 'intermediate' in exp_lower:
        return 3
    elif 'senior' in exp_lower or 'sr' in exp_lower:
        return 7
    elif 'lead' in exp_lower or 'principal' in exp_lower:
        return 10

    return np.nan

def standardize_education(edu):
    """Standardize education levels"""
    if pd.isna(edu):
        return edu

    edu_lower = str(edu).lower()

    if 'phd' in edu_lower or 'doctorate' in edu_lower:
        return 'PhD'
    elif 'master' in edu_lower or "master's" in edu_lower:
        return "Master's"
    elif 'bachelor' in edu_lower or "bachelor's" in edu_lower:
        return "Bachelor's"
    elif 'high school' in edu_lower or 'secondary' in edu_lower or '2nd level' in edu_lower:
        return '2nd Level'

    return edu

def categorize_job_field(job_title):
    """Categorize job titles into standardized fields"""
    if pd.isna(job_title):
        return 'Other'

    title_lower = str(job_title).lower()

    # Management
    if any(x in title_lower for x in ['manager', 'director', 'executive', 'ceo', 'cto', 'cfo']):
        return 'Management'

    # Research
    if any(x in title_lower for x in ['research', 'scientist', 'researcher']):
        return 'Research'

    # Data Science & Engineering
    if any(x in title_lower for x in ['data scientist', 'data engineer', 'ml engineer', 'machine learning']):
        return 'Data Science & Eng'

    # Software Engineering
    if any(x in title_lower for x in ['software', 'developer', 'programmer', 'engineer']):
        return 'Engineering'

    # Data Analysis
    if any(x in title_lower for x in ['data analyst', 'business analyst', 'analyst']):
        return 'Data Analyst'

    # Finance
    if any(x in title_lower for x in ['finance', 'financial', 'accountant']):
        return 'Finance'

    # Sales & Marketing
    if any(x in title_lower for x in ['sales', 'marketing', 'business development']):
        return 'Sales and Marketing'

    # HR
    if any(x in title_lower for x in ['hr', 'human resources', 'recruiter']):
        return 'Human Resources'

    # IT
    if any(x in title_lower for x in ['it ', 'information technology', 'system admin']):
        return 'IT'

    # Operations
    if any(x in title_lower for x in ['operations', 'project manager']):
        return 'Operations'

    # Consulting
    if 'consultant' in title_lower or 'consulting' in title_lower:
        return 'Consultancy'

    return 'Other'

# ============================================
# 5. OUTLIER DETECTION AND REMOVAL
# ============================================
def remove_outliers_iqr(df, column='Salary', multiplier=3):
    """Remove outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    initial_count = len(df)
    df_clean = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    removed_count = initial_count - len(df_clean)

    print(f"Removed {removed_count} outliers ({removed_count/initial_count*100:.1f}%)")
    print(f"Lower bound: ${lower_bound:,.0f}, Upper bound: ${upper_bound:,.0f}")

    return df_clean

# ============================================
# MAIN TRANSFORMATION PIPELINE
# ============================================
def transform_salary_data(df):
    """Complete data transformation pipeline"""
    print("Starting data transformation...")
    print(f"Initial records: {len(df)}")

    # 1. Remove duplicates
    print("\n1. Removing duplicates...")
    df = fuzzy_match_duplicates(df, threshold=90)
    print(f"Records after deduplication: {len(df)}")

    # 2. Handle missing values
    print("\n2. Handling missing values...")
    df = handle_missing_values(df)
    print(f"Records after missing value handling: {len(df)}")

    # 3. Standardization
    print("\n3. Standardizing data...")
    if 'Job Title' in df.columns:
        df['Job Title'] = df['Job Title'].apply(standardize_job_titles)

    if 'Location' in df.columns:
        df['Location'] = df['Location'].apply(standardize_locations)

    if 'Company' in df.columns:
        df['Company'] = df['Company'].apply(standardize_companies)

    # Normalize salary
    df['Salary'] = df.apply(normalize_salary, axis=1)

    # 4. Feature engineering
    print("\n4. Engineering features...")

    # Extract years of experience
    if 'Years of Experience' in df.columns:
        df['Years of Experience'] = df['Years of Experience'].apply(
            lambda x: extract_years_experience(x) if isinstance(x, str) else x
        )

    # Standardize education
    if 'Education Level' in df.columns:
        df['Education Level'] = df['Education Level'].apply(standardize_education)

    # Create job field category
    if 'Job Title' in df.columns:
        df['Job Field'] = df['Job Title'].apply(categorize_job_field)

    # 5. Remove outliers
    print("\n5. Removing outliers...")
    df = remove_outliers_iqr(df, column='Salary', multiplier=3)

    print(f"\nFinal records: {len(df)}")
    print("\nTransformation complete!")

    return df


In [16]:
# Apply transformation
df_clean = transform_salary_data(df)

# Save cleaned data
df_clean.to_csv('salary_data_cleaned.csv', index=False)
print("\nCleaned data saved to 'salary_data_cleaned.csv'")

Starting data transformation...
Initial records: 6704

1. Removing duplicates...
Records after deduplication: 6704

2. Handling missing values...
Records after missing value handling: 6699

3. Standardizing data...

4. Engineering features...

5. Removing outliers...
Removed 4 outliers (0.1%)
Lower bound: $-200,000, Upper bound: $430,000

Final records: 6695

Transformation complete!

Cleaned data saved to 'salary_data_cleaned.csv'


/var/folders/y4/0lcmny8s1696ft7y4dphy7kr0000gn/T/ipykernel_26606/1645941999.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Years of Experience'].fillna(median_exp, inplace=True)
/var/folders/y4/0lcmny8s1696ft7y4dphy7kr0000gn/T/ipykernel_26606/1645941999.py:47: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

In [17]:
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from scipy import stats

# Load your data
df = pd.read_csv('salary_data.csv')

# For demonstration, creating sample data structure

# ============================================
# 1. DUPLICATE REMOVAL (Fuzzy Matching)
# ============================================
def fuzzy_match_duplicates(df, threshold=90):
    """Remove fuzzy duplicates based on key columns"""
    df = df.copy()

    # Create comparison string for fuzzy matching
    if all(col in df.columns for col in ['Company', 'Job Title', 'Location', 'Salary']):
        df['compare_str'] = (df['Company'].fillna('') + '|' +
                            df['Job Title'].fillna('') + '|' +
                            df['Location'].fillna('') + '|' +
                            df['Salary'].astype(str))

        to_remove = set()
        for i in range(len(df)):
            if i in to_remove:
                continue
            for j in range(i+1, len(df)):
                if j in to_remove:
                    continue
                similarity = fuzz.ratio(df.iloc[i]['compare_str'],
                                       df.iloc[j]['compare_str'])
                if similarity >= threshold:
                    to_remove.add(j)

        df = df[~df.index.isin(to_remove)]
        df = df.drop('compare_str', axis=1)

    return df

# ============================================
# 2. MISSING VALUE HANDLING
# ============================================
def handle_missing_values(df):
    """Impute missing values using median/mode strategies"""
    df = df.copy()

    # Median imputation for experience (5% missing)
    if 'Years of Experience' in df.columns:
        median_exp = df['Years of Experience'].median()
        df['Years of Experience'].fillna(median_exp, inplace=True)

    # Mode imputation for education (8% missing)
    if 'Education Level' in df.columns:
        mode_edu = df['Education Level'].mode()[0] if not df['Education Level'].mode().empty else "Bachelor's"
        df['Education Level'].fillna(mode_edu, inplace=True)

    # Remove records with missing salary (core variable)
    if 'Salary' in df.columns:
        df = df.dropna(subset=['Salary'])

    return df

# ============================================
# 3. STANDARDIZATION
# ============================================
def standardize_job_titles(title):
    """Standardize job title variations"""
    if pd.isna(title):
        return title

    title_lower = str(title).lower()

    # Software Engineer variations
    if any(x in title_lower for x in ['swe', 'se ', 'software engineer', 'software eng']):
        return 'Software Engineer'

    # Data Analyst variations
    if any(x in title_lower for x in ['data analyst', 'data analysis']):
        return 'Data Analyst'

    # Manager variations
    if 'manager' in title_lower:
        if 'senior' in title_lower or 'sr' in title_lower:
            return 'Senior Manager'
        return 'Manager'

    return title.title()

def standardize_locations(location):
    """Standardize location variations"""
    if pd.isna(location):
        return location

    loc_lower = str(location).lower()

    # San Francisco variations
    if any(x in loc_lower for x in ['sf', 'san fran', 'san francisco']):
        return 'San Francisco'

    # New York variations
    if any(x in loc_lower for x in ['ny', 'nyc', 'new york']):
        return 'New York'

    return location.title()

def standardize_companies(company):
    """Standardize company name variations"""
    if pd.isna(company):
        return company

    comp_lower = str(company).lower()

    # Meta/Facebook variations
    if any(x in comp_lower for x in ['fb', 'facebook', 'meta']):
        return 'Meta'

    # Google/Alphabet variations
    if any(x in comp_lower for x in ['google', 'alphabet']):
        return 'Google'

    return company.title()

def normalize_salary(row):
    """Convert hourly/monthly salaries to annual"""
    salary = row['Salary']

    # Check if there's a salary type column
    if 'Salary Type' in row and pd.notna(row['Salary Type']):
        salary_type = str(row['Salary Type']).lower()

        if 'hourly' in salary_type or 'hour' in salary_type:
            # Assume 40 hours/week, 52 weeks/year
            return salary * 40 * 52
        elif 'monthly' in salary_type or 'month' in salary_type:
            return salary * 12

    # If salary seems too low, might be hourly or monthly
    if salary < 1000:  # Likely hourly
        return salary * 40 * 52
    elif 1000 <= salary < 15000:  # Likely monthly
        return salary * 12

    return salary

# ============================================
# 4. FEATURE ENGINEERING
# ============================================
def extract_years_experience(exp_str):
    """Extract years of experience from various formats"""
    if pd.isna(exp_str):
        return np.nan

    # If already a number
    if isinstance(exp_str, (int, float)):
        return int(round(exp_str))  # Convert to whole number

    exp_lower = str(exp_str).lower()

    # Handle ranges like "3-5 years" -> average (4)
    import re
    range_match = re.search(r'(\d+)\s*-\s*(\d+)', exp_lower)
    if range_match:
        return int(round((int(range_match.group(1)) + int(range_match.group(2))) / 2))

    # Handle single numbers
    num_match = re.search(r'\d+', exp_lower)
    if num_match:
        return int(num_match.group())

    # Handle text levels
    if 'entry' in exp_lower or 'junior' in exp_lower:
        return 1
    elif 'mid' in exp_lower or 'intermediate' in exp_lower:
        return 3
    elif 'senior' in exp_lower or 'sr' in exp_lower:
        return 7
    elif 'lead' in exp_lower or 'principal' in exp_lower:
        return 10

    return np.nan

def standardize_education(edu):
    """Standardize education levels"""
    if pd.isna(edu):
        return edu

    edu_lower = str(edu).lower()

    if 'phd' in edu_lower or 'doctorate' in edu_lower:
        return 'PhD'
    elif 'master' in edu_lower or "master's" in edu_lower:
        return "Master's"
    elif 'bachelor' in edu_lower or "bachelor's" in edu_lower:
        return "Bachelor's"
    elif 'high school' in edu_lower or 'secondary' in edu_lower or '2nd level' in edu_lower:
        return '2nd Level'

    return edu

def categorize_job_field(job_title):
    """Categorize job titles into standardized fields"""
    if pd.isna(job_title):
        return 'Other'

    title_lower = str(job_title).lower()

    # Management
    if any(x in title_lower for x in ['manager', 'director', 'executive', 'ceo', 'cto', 'cfo']):
        return 'Management'

    # Research
    if any(x in title_lower for x in ['research', 'scientist', 'researcher']):
        return 'Research'

    # Data Science & Engineering
    if any(x in title_lower for x in ['data scientist', 'data engineer', 'ml engineer', 'machine learning']):
        return 'Data Science & Eng'

    # Software Engineering
    if any(x in title_lower for x in ['software', 'developer', 'programmer', 'engineer']):
        return 'Engineering'

    # Data Analysis
    if any(x in title_lower for x in ['data analyst', 'business analyst', 'analyst']):
        return 'Data Analyst'

    # Finance
    if any(x in title_lower for x in ['finance', 'financial', 'accountant']):
        return 'Finance'

    # Sales & Marketing
    if any(x in title_lower for x in ['sales', 'marketing', 'business development']):
        return 'Sales and Marketing'

    # HR
    if any(x in title_lower for x in ['hr', 'human resources', 'recruiter']):
        return 'Human Resources'

    # IT
    if any(x in title_lower for x in ['it ', 'information technology', 'system admin']):
        return 'IT'

    # Operations
    if any(x in title_lower for x in ['operations', 'project manager']):
        return 'Operations'

    # Consulting
    if 'consultant' in title_lower or 'consulting' in title_lower:
        return 'Consultancy'

    return 'Other'

# ============================================
# 5. OUTLIER DETECTION AND REMOVAL
# ============================================
def remove_outliers_iqr(df, column='Salary', multiplier=3):
    """Remove outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    initial_count = len(df)
    df_clean = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    removed_count = initial_count - len(df_clean)

    print(f"Removed {removed_count} outliers ({removed_count/initial_count*100:.1f}%)")
    print(f"Lower bound: ${lower_bound:,.0f}, Upper bound: ${upper_bound:,.0f}")

    return df_clean

# ============================================
# MAIN TRANSFORMATION PIPELINE
# ============================================
def transform_salary_data(df):
    """Complete data transformation pipeline"""
    print("Starting data transformation...")
    print(f"Initial records: {len(df)}")

    # 1. Remove duplicates
    print("\n1. Removing duplicates...")
    df = fuzzy_match_duplicates(df, threshold=90)
    print(f"Records after deduplication: {len(df)}")

    # 2. Handle missing values
    print("\n2. Handling missing values...")
    df = handle_missing_values(df)
    print(f"Records after missing value handling: {len(df)}")

    # 3. Standardization
    print("\n3. Standardizing data...")
    if 'Job Title' in df.columns:
        df['Job Title'] = df['Job Title'].apply(standardize_job_titles)

    if 'Location' in df.columns:
        df['Location'] = df['Location'].apply(standardize_locations)

    if 'Company' in df.columns:
        df['Company'] = df['Company'].apply(standardize_companies)

    # Normalize salary
    df['Salary'] = df.apply(normalize_salary, axis=1)

    # 4. Feature engineering
    print("\n4. Engineering features...")

    # Extract years of experience
    if 'Years of Experience' in df.columns:
        df['Years of Experience'] = df['Years of Experience'].apply(
            lambda x: extract_years_experience(x) if isinstance(x, str) else x
        )
        # Ensure it's an integer
        df['Years of Experience'] = df['Years of Experience'].round().astype('Int64')

    # Ensure Age is an integer
    if 'Age' in df.columns:
        df['Age'] = df['Age'].round().astype('Int64')

    # Standardize education
    if 'Education Level' in df.columns:
        df['Education Level'] = df['Education Level'].apply(standardize_education)

    # Create job field category
    if 'Job Title' in df.columns:
        df['Job Field'] = df['Job Title'].apply(categorize_job_field)

    # 5. Remove outliers
    print("\n5. Removing outliers...")
    df = remove_outliers_iqr(df, column='Salary', multiplier=3)

    print(f"\nFinal records: {len(df)}")
    print("\nTransformation complete!")

    return df

# Apply transformation
df_clean = transform_salary_data(df)

# Save cleaned data
df_clean.to_csv('salary_data_cleaned.csv', index=False)
# print("\nCleaned data saved to 'salary_data_cleaned.csv'")

Starting data transformation...
Initial records: 6704

1. Removing duplicates...
Records after deduplication: 6704

2. Handling missing values...
Records after missing value handling: 6699

3. Standardizing data...

4. Engineering features...

5. Removing outliers...
Removed 4 outliers (0.1%)
Lower bound: $-200,000, Upper bound: $430,000

Final records: 6695

Transformation complete!


/var/folders/y4/0lcmny8s1696ft7y4dphy7kr0000gn/T/ipykernel_26606/2949198368.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Years of Experience'].fillna(median_exp, inplace=True)
/var/folders/y4/0lcmny8s1696ft7y4dphy7kr0000gn/T/ipykernel_26606/2949198368.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting